# **⚒️ Lectura de los Datos**

Este notebook tiene como objetivo procesar un libro en formato PDF, extraer su contenido como texto plano y separar dicho contenido por capítulos. Esta preparación facilita análisis posteriores, como procesamiento de lenguaje natural (NLP), análisis de sentimientos y otras tareas de exploración textual.

## **📚 Librerías**

In [2]:
import os # Para el manejo de archivos
import re # Para expresiones regulares
import fitz # Para la lectura del PDF 
import unicodedata # Para quitar tildes
from tqdm import tqdm # Para la barra de progreso

## **📦 Variables Globales**

In [2]:
pdf_file = '../Data/Raw/El_Ojo_Del_Mundo.pdf' # Ruta donde descargamos el PDF
text_file = '../Data/Processed/El_Ojo_Del_Mundo.txt' # Ruta donde guararemos el .txt

## **🧹 Limpieza de la data**

### **Transformación del PDF**

In [3]:
doc = fitz.open(pdf_file)

# Convertimos el archivo en .txt
with open(text_file, 'w', encoding='utf-8') as file:
    for page in tqdm(doc, desc='Extrayendo texto', unit='página'):
        text = page.get_text()
        file.write(text)

Extrayendo texto: 100%|██████████| 762/762 [00:02<00:00, 286.51página/s]


In [4]:
# Guardamos nuestro .txt
with open(text_file, 'r', encoding='utf-8') as file:
    text = file.read()

In [6]:
# Vista previa del texto
print(text[:500])

www.lectulandia.com - Página 1
 
El mundo gira y gira y una vez tras otra vuelven, al cabo de los años, de los
milenios, los días prometidos en las leyendas. Desde el horrible momento en
el que Lews Therin, que fue Señor de la Luz, Impulsado por las fuerzas de la
Oscuridad, dio muerte a todos los suyos, no ha cesado la lucha entre la luz y
las tinieblas, la vida y la muerte, el bien y el mal.
En una pequeña aldea solitaria de la región de Dos Rios vive Rand, un joven
granjero, en compañía de su 


In [7]:
# Eliminamos los pies de página
clean_text = re.sub(r"www\.lectulandia\.com\s*-\s*Página\s*\d+", '', text)

In [8]:
# Quitamos las tildes
clean_text = unicodedata.normalize('NFKD', clean_text).encode('ascii', 'ignore').decode('utf-8')

In [10]:
# Vista previa del texto limpio
print(clean_text[:500])


 
El mundo gira y gira y una vez tras otra vuelven, al cabo de los anos, de los
milenios, los dias prometidos en las leyendas. Desde el horrible momento en
el que Lews Therin, que fue Senor de la Luz, Impulsado por las fuerzas de la
Oscuridad, dio muerte a todos los suyos, no ha cesado la lucha entre la luz y
las tinieblas, la vida y la muerte, el bien y el mal.
En una pequena aldea solitaria de la region de Dos Rios vive Rand, un joven
granjero, en compania de su padre. Una noche son asaltados


### **Separación del texto por capitulos**

La idea es analizar de manera fragmentada como va evolucionando el contenido de la obra, ver si hay diferencias cuando tenemos POVs de diferentes personajes y ver si el tono es el mismo a lo largo de toda la obra.

In [11]:
# Búsqueda de los capítulos
patron_capitulos = r"\n\s*(\d{1,2})\s+([^\n]+)"
coincidencias = list(re.finditer(patron_capitulos, clean_text))

In [12]:
# Búsqueda del glosario
inicio_glosario = re.search('glosario', clean_text)

In [13]:
# Sabemos que el primer libro tiene 53 capítulos.
total_esperado = 53

if len(coincidencias) == total_esperado:
    print(f'El número de capítulos identificados coincide con el esperado: {len(coincidencias)}')
else:
    print(f'El número de capítulos identificados no coincide con el esperado: {len(coincidencias)}')

El número de capítulos identificados coincide con el esperado: 53


In [ ]:
# Guardamos el texto por capítulos
for i in range(len(coincidencias)):
    
    ini_cap = coincidencias[i].start()
    fin_cap = coincidencias[i + 1].start() if i + 1 < len(coincidencias) else len(clean_text)
    
    if fin_cap > inicio_glosario.start():
        fin_cap = inicio_glosario.start()

    titulo = coincidencias[i].group(2).replace(' ', '_').lower()
    
    num_archivo = i + 1 if len(str(i + 1)) == 2 else f'0{i+1}'
    nombre_archivo = f'{num_archivo}.{titulo}.txt'
    
    with open(f'../Data/Clean/01_EOdM/{nombre_archivo}', 'w', encoding='utf-8') as file:
        file.write(clean_text[ini_cap:fin_cap])

### **Creación de .txt con el contenido completo**

Aquí vamos a aprovechar que ya eliminamos todo el texto que no nos interesa (píes de página y glosario) para consolidar un .txt con la novela completa

In [ ]:
# Consolidación de capítulos
clean_path = '../Data/Clean/01_EOdM'

final_text = ''

for cap in os.listdir(clean_path):
    with open(os.path.join(clean_path, cap), 'r', encoding='utf-8') as file:
        texto = file.read()
    final_text += texto

In [ ]:
# Descargamos el archivo completo
with open('../Data/Clean/Completes/01_EOdM.txt', 'w', encoding='utf-8') as file:
    file.write(final_text)